# DDPG → SKRL: training semplicissimo (con segnale di stop)

Conversione del notebook originale (env "detector optimizer" + DDPG in `torchrl`) in **SKRL**.

Per avere un training *semplicissimo* e affidabile, l'ambiente e' stato semplificato rispetto all'originale, ma qui e' stato **reintrodotto il segnale di stop**:

- **Azione**: 2 componenti continue, entrambe normalizzate in `[-1, 1]`:
  - `action[0]` -> rimappata in `[50, 1000]` mm: posizione del prossimo piano.
  - `action[1]` -> se `> 0` l'agente decide di fermarsi (equivalente alla soglia 0.5 usata nell'originale su un range `[0, 1]`).
- Normalizzare entrambe le componenti nello stesso range evita che il rumore di esplorazione (stesso `std` su tutte le dimensioni, limite della `GaussianNoise` di SKRL) sia sbilanciato tra la posizione (range ampio, ~950 mm) e lo stop (range stretto).
- **Osservazione**: vettore a dimensione fissa e piccola (posizioni dei piani, paddate, + errore theta + errore beta), invece dei grandi array `(n_tracks, 20)` originali basati su un `Dict` di spazi.
- Resta comunque un limite di sicurezza (`MAX_PLANES`) che tronca l'episodio se l'agente non impara mai a fermarsi da solo.
- La fisica della simulazione (`simulate_tracks`) e la valutazione (calcolo di beta/theta ricostruiti) sono **identiche** all'originale.


## Simulazione del detector

Stesse funzioni `simulate_tracks` e valutazione (beta/theta ricostruiti) del notebook originale, solo `eval_sim` è stata estratta come funzione libera invece che come metodo privato della classe.

## Ambiente Gymnasium semplificato

- `action_space`: `Box(-1, 1, shape=(2,))` -> `action[0]` = posizione del prossimo piano (rimappata in `[50, 1000]`), `action[1]` = segnale di stop (soglia a 0).
- `observation_space`: `Box(shape=(MAX_PLANES + 2,))` -> posizioni dei piani (paddate a zero) + errore theta + errore beta.
- Episodio: termina (`terminated=True`) quando l'agente sceglie di fermarsi (e ha gia' piazzato almeno un piano); in alternativa viene troncato (`truncated=True`) dopo `MAX_PLANES` piani, come rete di sicurezza.
- Reward: stessa logica di penalita'/bonus su beta_dist/theta_dist dell'originale. Quando l'agente si ferma, quel passo non piazza un nuovo piano (come nell'originale).

In [9]:
from envs import SimpleRoTDetector

env = SimpleRoTDetector(n_tracks=200, spatial_res=0.1, time_res=0.3, tof_margin=50.0)

In [10]:
from skrl.models.torch import Model, DeterministicMixin
from torch import nn


class Actor(DeterministicMixin, Model):
    def __init__(self, observation_space, action_space, device):
        Model.__init__(self, observation_space=observation_space, action_space=action_space, device=device)
        DeterministicMixin.__init__(self, clip_actions=True)

        self.net = nn.Sequential(
            nn.Linear(self.num_observations, 2**10), nn.ReLU(),
            nn.Linear(2**10, 2**12), nn.ReLU(),
            nn.Linear(2**12, 2**10), nn.ReLU(),
            nn.Linear(2**10, self.num_actions), nn.Tanh(),
        )

    def compute(self, inputs, role=""):
        return self.net(inputs["observations"]), {}


class Critic(DeterministicMixin, Model):
    def __init__(self, observation_space, action_space, device):
        Model.__init__(self, observation_space=observation_space, action_space=action_space, device=device)
        DeterministicMixin.__init__(self, clip_actions=False)

        self.net = nn.Sequential(
            nn.Linear(self.num_observations + self.num_actions, 2**12), nn.ReLU(),
            nn.Linear(2**12, 2**10), nn.ReLU(),
            nn.Linear(2**10, 2**8), nn.ReLU(),
            nn.Linear(2**8, 1),
        )

    def compute(self, inputs, role=""):
        x = torch.cat([inputs["observations"], inputs["taken_actions"]], dim=-1)
        return self.net(x), {}


In [11]:
from skrl.envs.wrappers.torch import wrap_env
from skrl.memories.torch import RandomMemory
from skrl.resources.noises.torch import GaussianNoise
from skrl.agents.torch.ddpg import DDPG, DDPG_CFG

device = "cpu"

env = wrap_env(env)
memory = RandomMemory(memory_size=20000, num_envs=env.num_envs, device=device)

models = {
    "policy": Actor(env.observation_space, env.action_space, device),
    "target_policy": Actor(env.observation_space, env.action_space, device),
    "critic": Critic(env.observation_space, env.action_space, device),
    "target_critic": Critic(env.observation_space, env.action_space, device),
}

cfg = DDPG_CFG(
    batch_size=32,
    discount_factor=0.98,
    polyak=0.01,
    learning_rate=1e-4,
    random_timesteps=0,
    learning_starts=0,
    exploration_noise=GaussianNoise,
    exploration_noise_kwargs={"mean": 0.0, "std": 0.8, "device": device},
)
cfg.experiment.write_interval = 20
cfg.experiment.checkpoint_interval = 0
cfg.experiment.experiment_name = "ddpg_RoT-stable"
cfg.experiment.directory = "./runs"
cfg.close_environment_at_exit = True
cfg.timesteps = 10000
cfg.disable_progressbar = False
cfg.headless = True
cfg.environment_info = True

agent = DDPG(
    models=models,
    memory=memory,
    observation_space=env.observation_space,
    action_space=env.action_space,
    device=device,
    cfg=cfg,
)


[skrl:INFO] Environment wrapper: 'auto' (class: gymnasium.core.Env)
[skrl:INFO] Environment wrapper: Gymnasium
[skrl:WARNING] Failed to check for a vectorized environment: module 'gymnasium.experimental' has no attribute 'vector'


In [ ]:
from torch.utils.tensorboard import SummaryWriter
from skrl.trainers.torch import StepTrainer
import numpy as np

#Make custom trainer more refined, add visualization of detector config distrubution and other data
#Write description on tensorboard data, Qvalue, timesteps and so on, comment and explanations
#Write different versions of the whole system, such as Reward on Termination, and potential reward shaping, and so on
class customTrainer(StepTrainer):
    def __init__(self, env, agent, cfg):
        super().__init__(env=env, agents=agent, cfg=cfg)
        self.writer = SummaryWriter(log_dir="./runs")

    def train(self):
        self.best_reward = -np.inf

        for timestep in range(self.cfg.timesteps):
            super().train()
            
            obs = env._obs_and_metrics()
            reward, cost = env.calculate_reward(obs[1], obs[2])
            
            #print best detector state and reward if reward is better than previous best
            if reward > self.best_reward:
                self.best_reward = reward
                if(env.detector_state is not None and len(env.detector_state) > 1):
                    self.writer.add_figure("Best Detector State", env.draw_detector(n_planes=len(env.detector_state), z_planes=env.detector_state, plane_size=100.0, tof_margin=50, x_hits=obs[3], y_hits=obs[4], n_tracks_show=env.n_tracks, label=f"Detector 3D - Reward: {reward:.2f}, Planes: {len(env.detector_state)}\ndist theta: {obs[1]:.2f}, dist beta: {obs[2]:.2f}\ncost: {cost:.2f}"), global_step=timestep)

In [ ]:
from skrl.trainers.torch import SequentialTrainer, SequentialTrainerCfg
trainer = SequentialTrainer(env=env, agents=agent, cfg=cfg)
trainer.train()